# Tumor Stroma Analysis
##### Franziska Niemeyer, 2025-05-23

In [ ]:
import os
import warnings

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import anndata as ad
import numpy as np
import seaborn as sns
import seaborn.objects as so
from matplotlib.colors import LinearSegmentedColormap
import decoupler as dc
from aquarel import load_theme

from cnv_inference.infercnv_subclusters import extract_subclusters, add_subclusters_to_adata

sc.set_figure_params(color_map="viridis_r", dpi_save=600, vector_friendly=True, fontsize=12)
color_palette = "Set1"

warnings.filterwarnings(action="ignore")

In [ ]:
plt.rcdefaults()
plt.rcParams.update({
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.linewidth":     0.8,
    "xtick.major.size":   3,
    "ytick.major.size":   3,
    "xtick.labelsize":    7,
    "ytick.labelsize":    7,
    "axes.titlesize":     8,
    "axes.titleweight":   "bold",
    "axes.labelsize":     7,
    "figure.dpi":         300,
})

In [ ]:
ADATA       = "../../../../quality_control/external-cohort/pre-processing/adata.h5ad"
OUT_DIR     = "output"
FIG_DIR     = "figures"

sc.settings.figdir = FIG_DIR

### Import data and annotations

In [ ]:
adata = ad.read_h5ad(ADATA)
adata

In [ ]:
adata.obs['histology'].value_counts()

In [ ]:
adata.obs['outcome'].value_counts()

In [ ]:
print(adata.obs['outcome'].dtype)

In [ ]:
# Collapse the four fine-grained outcome groups into Long / Short
adata.obs['outcome'] = adata.obs['outcome'].astype(str).replace({
    'HC-short': 'Short',
    'LC-short': 'Short',
    'HC-long':  'Long',
    'LC-long':  'Long',
    'control tissue': 'control tissue',
}).astype('category')
adata.obs['outcome'].value_counts()

### Filter data

In [ ]:
OBSERVATION_HISTOLOGY = ["Tumor stroma", "Tumor epithelium"]
REFERENCE_HISTOLOGY   = "Ovarian stroma"

In [ ]:
reference = adata[adata.obs['histology'] == REFERENCE_HISTOLOGY].copy()
reference.obs['anno'] = reference.obs['histology'].astype(str)
print(f"{reference.n_obs} spots in reference ({REFERENCE_HISTOLOGY})")

In [ ]:
observations = adata[adata.obs['histology'].isin(OBSERVATION_HISTOLOGY)].copy()
observations.obs['anno'] = (
    observations.obs['histology'].astype(str)
    + ' '
    + observations.obs['patient'].astype(str)
)
print(f"{observations.n_obs} spots in observations ({OBSERVATION_HISTOLOGY})")

In [ ]:
adata_new = ad.concat([reference, observations], join='outer')
adata_new.obs['anno'].value_counts()

Filter to a specific patient

In [ ]:
PATIENT_FILTER = 'OVA09'

if PATIENT_FILTER is not None:
    keep = (
        (adata_new.obs['anno'] == REFERENCE_HISTOLOGY) |
        (adata_new.obs['anno'].str.contains(PATIENT_FILTER))
    )
    adata_new = adata_new[keep].copy()

adata_new.obs['anno'].value_counts()

In [ ]:
annos = adata_new.obs.reset_index(names='Barcode')
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)
annos[['Barcode', 'anno']].to_csv(os.path.join(OUT_DIR, "annotations.csv"), index=False)

### Add CNV data to adata

In [ ]:
hmm_out   = "../infercnv_runs/tumor-stroma-ova09"

In [ ]:
result = extract_subclusters(
    infercnv_dir=hmm_out,
    level="regions",
    reference_pattern="Ovarian stroma",
)

adata_new = add_subclusters_to_adata(adata_new, result, obs_key="Clone")

# Shorten the subcluster labels
adata_new.obs["Clone"] = adata_new.obs["Clone"].astype(str).str.replace(
    r"(.*) - (.*?)_s(\d+)$", 
    lambda m: f"{m.group(2)} s{m.group(3)}", 
    regex=True
).replace("nan", "Stroma")

adata_new.obs["Clone"].value_counts()

In [ ]:
mask = adata_new.obs['histology'] == 'Ovarian stroma'
adata_new.obs['Clone'] = adata_new.obs['Clone'].astype(str)
adata_new.obs.loc[mask, 'Clone'] = 'Stroma'

adata_new.obs['Clone'] = pd.Categorical(adata_new.obs['Clone'])
print(adata_new.obs['Clone'].value_counts())

In [ ]:
adata_new.obs["Clone"] = (
    adata_new.obs["Clone"]
    .astype(str)
    .str.replace(r".*_s(\d+)$", r"s\1", regex=True)
    .replace("nan", "Other")   # unmatched spots = reference/other patients
)

adata_new.obs["Clone"].value_counts()

In [ ]:
adata_sub = adata_new[~adata_new.obs["Clone"].isin(["Other", "s5"])].copy()

min_cells = 2
valid_clones = adata_sub.obs["Clone"].value_counts()
valid_clones = valid_clones[valid_clones >= min_cells].index
adata_sub = adata_sub[adata_sub.obs["Clone"].isin(valid_clones)].copy()

sc.pp.highly_variable_genes(adata_sub, n_top_genes=5000, flavor='seurat_v3', subset=False, layer='counts')
sc.tl.rank_genes_groups(adata_sub, layer='log-transformed', groupby='Clone', method='wilcoxon', use_raw=False)

In [ ]:
stats_clone_D = sc.get.rank_genes_groups_df(adata_sub, 'Tumor stroma s4', key='rank_genes_groups')
stats_clone_D

In [ ]:
sc.pp.pca(adata_sub)
sc.pp.neighbors(adata_sub)
sc.tl.leiden(adata_sub, key_added='cluster', resolution=0.5, flavor='igraph')
sc.tl.umap(adata_sub)

In [ ]:
sc.pl.umap(adata_sub, color=['Clone', 'cluster'], wspace=0.5)

### Gene set enrichment analysis

In [ ]:
hallmark = dc.op.hallmark(organism='human')

# Filter by geneset size
geneset_size   = hallmark.groupby('source', observed=True).size()
gsea_genesets  = geneset_size.index[(geneset_size > 15) & (geneset_size < 500)]
hallmark_filt  = hallmark[hallmark['source'].isin(gsea_genesets)].copy()

print(hallmark_filt)

In [ ]:
adata_sub.uns['spatial'] = adata.uns['spatial'].copy()

In [ ]:
# ULM with Hallmark gene sets
dc.mt.ulm(data=adata_sub, net=hallmark_filt)
score = dc.pp.get_obsm(adata=adata_sub, key='score_ulm')
score

In [ ]:
print(adata_sub.obs[["sample", "patient"]].drop_duplicates())

In [ ]:
import squidpy as sq

tf = "COMPLEMENT"
sample = "HC-TMA3"
score.obsm["spatial"] = adata_sub.obsm["spatial"][
    adata_sub.obs.index.get_indexer(score.obs_names)
]

score_sub_spatial = score[
    (score.obs["sample"] == sample) &
    # (score.obs["Clone"] != "Stroma") &
    (score.obs["Clone"].count() > 1) &
    (score.obs["Clone"].notna())
].copy()

# Crop to spots present in the subset
score_sub_spatial = score_sub_spatial[score_sub_spatial.obs['histology'] != 'Ovarian stroma'].copy()
score_sub_spatial.obs['Clone'] = score_sub_spatial.obs['Clone'].cat.remove_unused_categories()
score_sub_spatial.obs['histology'] = score_sub_spatial.obs['histology'].cat.remove_unused_categories()

coords = score_sub_spatial.obsm["spatial"]
x_min, x_max = coords[:, 0].min(), coords[:, 0].max()
y_min, y_max = coords[:, 1].min(), coords[:, 1].max()
pad = 200  # padding around the spots

In [ ]:
score_sub.obs['Clone'].value_counts()

In [ ]:
clone_palette = {
    "Stroma": "#808080",
    # Tumor stroma — blues/teals with more contrast
    "Tumor stroma s1": "#08306B",
    "Tumor stroma s2": "#2171B5",
    "Tumor stroma s3": "#6BAED6",
    "Tumor stroma s4": "#BDD7E7",
    # Tumor epithelium — reds/oranges
    "Tumor epithelium s1": "#B2182B",
    "Tumor epithelium s2": "#E5594D",
    "Tumor epithelium s3": "#F4A582",
}

score_sub.obs["Clone"] = score_sub.obs["Clone"].astype("category")
score_sub.uns["Clone_colors"] = [
    clone_palette[c] for c in score_sub.obs["Clone"].cat.categories
    if c in clone_palette
]

In [ ]:
tf = "INTERFERON_ALPHA_RESPONSE"

# same order as spatial plot
clone_order = [c for c in clone_palette.keys() if c in score.obs["Clone"].cat.categories]
score_plot  = score[score.obs["Clone"].isin(clone_order)].copy()
score_plot.obs["Clone"] = pd.Categorical(score_plot.obs["Clone"], categories=clone_order, ordered=True)

fig, ax = plt.subplots(figsize=(4, 2), dpi=300)

sc.pl.violin(
    score_plot,
    keys=tf,
    groupby="Clone",
    order=clone_order,
    palette=clone_palette,
    rotation=90,
    ylabel="Enrichment score",
    stripplot=False,
    inner="box",
    show=False,
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("Enrichment score", fontsize=10)
ax.set_title('Interferon alpha response', fontsize=11, fontweight="bold")
ax.axhline(0, color="black", linewidth=0.5, linestyle="--", alpha=0.4)

# Separate stroma and epithelium groups with a vertical line
n_stroma = sum(1 for c in clone_order if "stroma" in c.lower())

plt.savefig(
    os.path.join(OUT_DIR, "interferon_alpha_violin.pdf"),
    bbox_inches="tight",
    dpi=600,
)
plt.savefig(
    os.path.join(OUT_DIR, "interferon_alpha_violin.png"),
    bbox_inches="tight",
    dpi=600,
)
plt.show()

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

fig, axes = plt.subplots(1, 3, figsize=(12, 5), dpi=300)

histology_categories = score_sub.obs["histology"].cat.categories.tolist()

def compartment_color(label):
    l = label.lower()
    if "stroma" in l:
        return "#3b6fb0"      # blue
    if "epi" in l:
        return "#d94a27"      # orange
    return "#999999"

histology_colors = [compartment_color(c) for c in histology_categories]
histology_cmap   = ListedColormap(histology_colors)

sq.pl.spatial_scatter(
    score_sub,
    color="histology",
    size=1.5,
    palette=histology_cmap,
    crop_coord=(x_min - pad, y_min - pad, x_max + pad, y_max + pad),
    library_id=sample,
    title="",
    legend_loc=None,
    ax=axes[1],
)
axes[1].set_title("Tissue compartment", fontsize=10, fontweight="bold")
axes[1].axis("off")

sq.pl.spatial_scatter(
    score_sub,
    color=None,
    size=1.5,
    crop_coord=(x_min - pad, y_min - pad, x_max + pad, y_max + pad),
    library_id=sample,
    title="",
    legend_loc=None,
    ax=axes[0],
)
axes[0].set_title("H&E", fontsize=10, fontweight="bold")
axes[0].axis("off")

sq.pl.spatial_scatter(
    score_sub,
    color="Clone",
    size=1.5,
    crop_coord=(x_min - pad, y_min - pad, x_max + pad, y_max + pad),
    library_id=sample,
    title="",
    legend_loc=None,
    ax=axes[2],
)
axes[2].set_title("CNV clones", fontsize=10, fontweight="bold")
axes[2].axis("off")

for ax in axes:
    ax.set_xlabel("")
    ax.set_ylabel("")

# Histology legend
histology_handles = [
    mpatches.Patch(color=c, label=l)
    for l, c in zip(histology_categories, histology_colors)
]

# Clone legend
clone_handles = [
    mpatches.Patch(color=color, label=clone)
    for clone, color in clone_palette.items()
    if clone in score_sub.obs["Clone"].cat.categories
]

# Place legends centered below each panel
axes[1].legend(
    handles=histology_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.02),
    borderaxespad=0,
    frameon=False,
    fontsize=7,
    title="Compartment",
    title_fontsize=8,
    ncol=1,
)
axes[2].legend(
    handles=clone_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.02),
    borderaxespad=0,
    frameon=False,
    fontsize=7,
    title="Clone",
    title_fontsize=8,
    ncol=2,
)

plt.tight_layout()
plt.savefig(
    os.path.join(OUT_DIR, "spatial_three_panel.pdf"),
    bbox_inches="tight",
    dpi=600,
)
plt.savefig(
    os.path.join(OUT_DIR, "spatial_three_panel.png"),
    bbox_inches="tight",
    dpi=600,
)
plt.show()

In [ ]:
df = dc.tl.rankby_group(
    adata=score,
    groupby='Clone',
    reference='rest',
    method='t-test_overestim_var',
)
df = df[df['stat'] > 0]
df

In [ ]:
n_markers = 3
source_markers = (
    df
    .groupby('group')
    .head(n_markers)
    .drop_duplicates('name')
    .groupby('group')['name']
    .apply(list)
    .to_dict()
)
source_markers.pop('B', None)
source_markers

In [ ]:
plt.rcdefaults()
plt.rcParams.update({
    'font.size':        10,
    'axes.labelsize':   10,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   9,
    'figure.titlesize': 12,
})

colors      = sns.diverging_palette(250, 30, l=65, s=80, center='light', n=256)
orange_cmap = LinearSegmentedColormap.from_list('orange_seq', colors[128:])

plt.figure(figsize=(12, 8))
g = sc.pl.matrixplot(
    adata=score,
    var_names=source_markers,
    groupby='Clone',
    dendrogram=False,
    standard_scale='var',
    colorbar_title='Z-scaled scores',
    cmap=orange_cmap,
    swap_axes=True,
    show=False,
)

for i, ax in enumerate(plt.gcf().get_axes()):
    print(f"\n--- Axes {i} ---")
    for artist in ax.get_children():
        if hasattr(artist, 'get_text') and artist.get_text():
            print("Text:", artist.get_text())
        if hasattr(artist, 'get_paths'):
            print("Patch/Line:", type(artist).__name__)

# remove bracket labels
fig = plt.gcf()
for artist in fig.get_axes()[1].get_children():
    if hasattr(artist, 'get_text'):
        artist.set_visible(False)

# Clean up y-tick labels
ax = list(g.values())[0]
yticks  = ax.get_yticks()
ylabels = [l.get_text().replace('_', ' ').title() for l in ax.get_yticklabels()]
ax.set_yticks(yticks)
ax.set_yticklabels(ylabels)

for ax in g.values():
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(labelsize=9)
    if ax.get_xlabel():
        ax.set_xlabel(ax.get_xlabel(), fontsize=10)
    if ax.get_ylabel():
        ax.set_ylabel(ax.get_ylabel(), fontsize=10)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontsize(9)

# Gene group axis border
for key, ax in g.items():
    if 'gene_group_ax' in key.lower():
        for patch in ax.patches:
            patch.set_linewidth(0.5)
        break

for key, ax in g.items():
    if 'groupby_ax' in key.lower():
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        break

# Reposition colourbar
for key, ax in g.items():
    if 'color_legend_ax' in key.lower():
        ax.remove()
        break

for key, ax in g.items():
    if 'groupby_ax' in key.lower():
        for artist in ax.get_children():
            artist.set_visible(False)
        ax.set_visible(False)
        break

fig.suptitle("Top enriched pathways per CNV clone", fontsize=12, fontweight="bold", y=.85, x=0.35)

fig     = plt.gcf()
main_ax = next(iter(g.values()))
im      = main_ax.collections[0]
cbar_ax = fig.add_axes([0.62, 0.15, 0.03, 0.5])
cbar    = fig.colorbar(im, cax=cbar_ax, orientation='vertical')
cbar.set_label('Z-scaled enrichment score')

plt.savefig(os.path.join(OUT_DIR, 'clones_matrixplot.png'), dpi=600, bbox_inches='tight')
plt.savefig(os.path.join(OUT_DIR, 'clones_matrixplot.pdf'), dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

genes = ['IFI27', 'C3', 'BST2']

clone_order = [c for c in clone_palette.keys()
               if c in adata_plot.obs['Clone'].cat.categories]
sp = adata_plot[adata_plot.obs['Clone'].isin(clone_order)].copy()
sp.obs['Clone'] = pd.Categorical(sp.obs['Clone'], categories=clone_order, ordered=True)

def expr(gene):
    x = sp[:, gene].layers['log-transformed']
    return np.asarray(x.todense()).ravel() if hasattr(x, 'todense') else np.asarray(x).ravel()

med = {}
for gene in genes:
    v = expr(gene)
    for c in clone_order:
        m = (sp.obs['Clone'].values == c)
        med[(gene, c)] = np.median(v[m])

vmin, vmax = 0, max(med.values())          # or set vmax=30 to match earlier
cmap = orange_cmap                          # your existing orange colormap
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

fig, axes = plt.subplots(
    len(genes), 1, figsize=(4, 0.8 * len(genes)), dpi=300,
    sharex=True, sharey=False, gridspec_kw={'hspace': 0},
)

for ax, gene in zip(axes, genes):
    # build a per-clone palette from that gene's medians
    gene_palette = {c: cmap(norm(med[(gene, c)])) for c in clone_order}
    sc.pl.violin(
        sp, keys=gene, groupby='Clone', order=clone_order,
        palette=gene_palette, stripplot=False, inner='box',
        show=False, ax=ax,
    )
    ax.set_xlabel(''); ax.set_title('')
    ax.set_ylabel(gene, fontsize=9, rotation=0, ha='right', va='center', labelpad=8)
    ax.tick_params(left=False, labelleft=False)
    for s in ax.spines.values():
        s.set_visible(False)

# continuous outer border via touching spines
axes[0].spines['top'].set_visible(True)
axes[-1].spines['bottom'].set_visible(True)
for ax in axes:
    ax.spines['left'].set_visible(True)
    ax.spines['right'].set_visible(True)

axes[-1].tick_params(labelbottom=True)
for label in axes[-1].get_xticklabels():
    label.set_rotation(90)

# colorbar legend
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
cbar = fig.colorbar(sm, ax=axes, fraction=0.03, pad=0.02)
cbar.set_label('Median normalized\nexpression', fontsize=9)
cbar.ax.tick_params(labelsize=8)

fig.suptitle('Genes of interest', fontsize=11, fontweight='bold', y=0.98)

plt.savefig(os.path.join(OUT_DIR, 'goi_stacked_violin.pdf'), bbox_inches='tight', dpi=600)
plt.savefig(os.path.join(OUT_DIR, 'goi_stacked_violin.png'), bbox_inches='tight', dpi=600)
plt.show()